# El Reto Kafka

## Instalación

In [406]:
# %pip install -q transformers torch pandas

## Constantes y Librerias

In [407]:
from contextlib import redirect_stdout
from pathlib import Path
from tabulate import tabulate
import torch
from transformers import AutoTokenizer, AutoModel

In [408]:
MODEL_DIR = Path("models")
RESULTS_DIR = Path("results")
CORPUS_PATH = Path("data/corpus.txt")
MODEL_NAME = "bert-base-multilingual-cased"
SENTENCES_LIMIT = 1
TOP_K = 5

## Funciones auxiliares

In [409]:
def getModel(model_name, model_dir):
  
  local_path= model_dir / model_name
  
  if not local_path.exists():
    
    local_path.mkdir(parents=True, exist_ok=True)
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    
    tokenizer.save_pretrained(local_path)
    model.save_pretrained(local_path)
    
  return local_path

In [410]:
def getTokenizer(model_path):
  return AutoTokenizer.from_pretrained(model_path)

In [411]:
def loadModel(model_path):
    model = AutoModel.from_pretrained(
        model_path,
        output_attentions=True
    )
    model.eval()
    return model

In [412]:
def getCorpus(corpus_path):
  with open(corpus_path, "r", encoding="utf-8") as file:
    return [line.strip() for line in file if line.strip()]

In [413]:
def getTokens(sentence, tokenizer):
    return tokenizer.tokenize(sentence)

In [414]:
def getTokens(sentence, tokenizer):
    encoded = tokenizer(sentence, add_special_tokens=True)
    return tokenizer.convert_ids_to_tokens(encoded["input_ids"])

In [415]:
def tokenizeCorpus(corpus, tokenizer):
    return [
        {
            "sentence": sentence,
            "tokens": getTokens(sentence, tokenizer)
        }
        for sentence in corpus
    ]

In [416]:
def saveResult(function, log_name, log_dir):
    log_path = Path(log_dir) / log_name
    log_path.parent.mkdir(parents=True, exist_ok=True)

    with open(log_path, "w", encoding="utf-8") as file:
        with redirect_stdout(file):
            function()

    return log_path

In [417]:
model_path = getModel(model_name=MODEL_NAME, model_dir=MODEL_DIR)

tokenizer = getTokenizer(model_path)
model = loadModel(model_path)

corpus = getCorpus(corpus_path=CORPUS_PATH)

tokenized_corpus = tokenizeCorpus(corpus, tokenizer)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## Parte A 

In [418]:
def printTokenizedCorpus(tokenized_corpus):
    for item in tokenized_corpus:
        print("Oración:", item["sentence"])
        print("Tokens:", item["tokens"])
        print()

In [419]:
saveResult(
    lambda: printTokenizedCorpus(tokenized_corpus),
    log_name="tokenization.txt",
    log_dir=RESULTS_DIR
)

WindowsPath('results/tokenization.txt')

In [420]:
printTokenizedCorpus(tokenized_corpus[:SENTENCES_LIMIT])

Oración: De no haberle parecido oportuna tal medida, ella misma hubiera
Tokens: ['[CLS]', 'De', 'no', 'haber', '##le', 'parecido', 'op', '##ort', '##una', 'tal', 'medida', ',', 'ella', 'misma', 'hubiera', '[SEP]']



## Parte B

In [421]:
def getAttentions(sentence, tokenizer, model):
    inputs = tokenizer(
        sentence,
        return_tensors="pt",
        add_special_tokens=True
    )

    with torch.no_grad():
        outputs = model(**inputs)

    return outputs.attentions

In [422]:
def printAttentionShapes(corpus, tokenizer, model):
    for sentence in corpus:
        attentions = getAttentions(sentence, tokenizer, model)

        print("Oración:", sentence)
        print("Número de capas:", len(attentions))

        for i, attention in enumerate(attentions):
            print(
                f"Capa {i}:",
                tuple(attention.shape)
            )

        print()

In [423]:
saveResult(
    lambda: printAttentionShapes(corpus, tokenizer, model),
    log_name="attention_shapes.txt",
    log_dir=RESULTS_DIR
)

WindowsPath('results/attention_shapes.txt')

In [424]:
printAttentionShapes(corpus[:SENTENCES_LIMIT], tokenizer, model)

Oración: De no haberle parecido oportuna tal medida, ella misma hubiera
Número de capas: 12
Capa 0: (1, 12, 16, 16)
Capa 1: (1, 12, 16, 16)
Capa 2: (1, 12, 16, 16)
Capa 3: (1, 12, 16, 16)
Capa 4: (1, 12, 16, 16)
Capa 5: (1, 12, 16, 16)
Capa 6: (1, 12, 16, 16)
Capa 7: (1, 12, 16, 16)
Capa 8: (1, 12, 16, 16)
Capa 9: (1, 12, 16, 16)
Capa 10: (1, 12, 16, 16)
Capa 11: (1, 12, 16, 16)



## Parte C

In [425]:
def getTopAttention( sentence, target_token, layer, head, tokenizer, model, top_n):
    inputs = tokenizer(
        sentence,
        return_tensors="pt",
        add_special_tokens=True
    )

    tokens = tokenizer.convert_ids_to_tokens(
        inputs["input_ids"][0]
    )

    with torch.no_grad():
        outputs = model(**inputs)

    attentions = outputs.attentions

    # Validar que el token exista
    if target_token not in tokens:
        raise ValueError(
            f"El token '{target_token}' no existe en: {tokens}"
        )

    token_index = tokens.index(target_token)

    # attention[layer]:
    # (batch, heads, tokens, tokens)
    #
    # Seleccionamos:
    # batch 0
    # cabeza indicada
    # fila del token que estamos analizando
    weights = attentions[layer][0, head, token_index]

    top_values, top_indices = torch.topk(
        weights,
        k=min(top_n, len(tokens))
    )

    results = []

    for position, (index, weight) in enumerate(
        zip(top_indices.tolist(), top_values.tolist()),
        start=1
    ):
        results.append([
            position,
            tokens[index],
            index,
            weight
        ])

    return results

In [426]:
def printTopAttention(sentence, target_token, layer, head, tokenizer, model, top_n):
    results = getTopAttention(
        sentence=sentence,
        target_token=target_token,
        layer=layer,
        head=head,
        tokenizer=tokenizer,
        model=model,
        top_n=top_n
    )

    print("Oración:", sentence)
    print("Token analizado:", target_token)
    print("Capa:", layer)
    print("Cabeza:", head)

    print(
        tabulate(
            results,
            headers=[
                "Ranking",
                "Token",
                "Índice",
                "Atención"
            ],
            tablefmt="grid",
            floatfmt=".6f"
        )
    )

    print()

In [427]:
def printAttentionAnalysis(sentence, target_tokens, layers, heads, tokenizer, model, top_n):
    print("=" * 80)
    print("Oración:", sentence)
    print("Tokens:", getTokens(sentence, tokenizer))
    print("=" * 80)
    print()

    for target_token in target_tokens:
        for layer in layers:
            for head in heads:

                printTopAttention(
                    sentence=sentence,
                    target_token=target_token,
                    layer=layer,
                    head=head,
                    tokenizer=tokenizer,
                    model=model,
                    top_n=top_n
                )

In [428]:
sentence = corpus[13]

target_tokens = [
    "madre",
    "Gregorio"
]

layers = [
    2,
    8
]

heads = [
    0,
    5
]

In [429]:
saveResult(
    lambda: printAttentionAnalysis(
        sentence=sentence,
        target_tokens=target_tokens,
        layers=layers,
        heads=heads,
        tokenizer=tokenizer,
        model=model,
        top_n=TOP_K
    ),
    log_name="attention_analysis.txt",
    log_dir=RESULTS_DIR
)

WindowsPath('results/attention_analysis.txt')

In [430]:
printAttentionAnalysis(
    sentence=sentence,
    target_tokens=target_tokens,
    layers=layers,
    heads=heads,
    tokenizer=tokenizer,
    model=model,
    top_n=TOP_K
)

Oración: La madre había querido visitar a Gregorio enseguida, pero el padre y la
Tokens: ['[CLS]', 'La', 'madre', 'había', 'quer', '##ido', 'visitar', 'a', 'Gregorio', 'ens', '##egu', '##ida', ',', 'pero', 'el', 'padre', 'y', 'la', '[SEP]']

Oración: La madre había querido visitar a Gregorio enseguida, pero el padre y la
Token analizado: madre
Capa: 2
Cabeza: 0
+-----------+---------+----------+------------+
|   Ranking | Token   |   Índice |   Atención |
+===========+=========+==========+============+
|         1 | [SEP]   |       18 |   0.180559 |
+-----------+---------+----------+------------+
|         2 | [CLS]   |        0 |   0.177115 |
+-----------+---------+----------+------------+
|         3 | el      |       14 |   0.084793 |
+-----------+---------+----------+------------+
|         4 | La      |        1 |   0.069662 |
+-----------+---------+----------+------------+
|         5 | pero    |       13 |   0.056462 |
+-----------+---------+----------+------------+

Oración: La

## Parte D

In [431]:
def compareAttention(sentence_1, sentence_2, target_token, layer, head, tokenizer, model, top_n):
    result_1 = getTopAttention(
        sentence_1,
        target_token,
        layer,
        head,
        tokenizer,
        model,
        top_n
    )

    result_2 = getTopAttention(
        sentence_2,
        target_token,
        layer,
        head,
        tokenizer,
        model,
        top_n
    )

    print("=" * 80)
    print(f"COMPARACIÓN DEL TOKEN: {target_token}")
    print(f"Capa: {layer} | Cabeza: {head}")
    print("=" * 80)

    print("\nORACIÓN 1:")
    print(sentence_1)
    print(tabulate(
        result_1,
        headers=["Ranking", "Token", "Índice", "Atención"],
        tablefmt="grid",
        floatfmt=".6f"
    ))

    print("\nORACIÓN 2:")
    print(sentence_2)
    print(tabulate(
        result_2,
        headers=["Ranking", "Token", "Índice", "Atención"],
        tablefmt="grid",
        floatfmt=".6f"
    ))

In [432]:
sentence_1 = corpus[13]
sentence_2 = corpus[17]

In [433]:
saveResult(
    lambda: compareAttention(
        sentence_1=sentence_1,
        sentence_2=sentence_2,
        target_token="Gregorio",
        layer=8,
        head=5,
        tokenizer=tokenizer,
        model=model,
        top_n=TOP_K
    ),
    log_name="attention_comparison.txt",
    log_dir=RESULTS_DIR
)

WindowsPath('results/attention_comparison.txt')

In [434]:
compareAttention(
    sentence_1=sentence_1,
    sentence_2=sentence_2,
    target_token="Gregorio",
    layer=8,
    head=5,
    tokenizer=tokenizer,
    model=model,
    top_n=TOP_K
)

COMPARACIÓN DEL TOKEN: Gregorio
Capa: 8 | Cabeza: 5

ORACIÓN 1:
La madre había querido visitar a Gregorio enseguida, pero el padre y la
+-----------+----------+----------+------------+
|   Ranking | Token    |   Índice |   Atención |
+===========+==========+==========+============+
|         1 | Gregorio |        8 |   0.883683 |
+-----------+----------+----------+------------+
|         2 | [CLS]    |        0 |   0.034807 |
+-----------+----------+----------+------------+
|         3 | la       |       17 |   0.024829 |
+-----------+----------+----------+------------+
|         4 | [SEP]    |       18 |   0.015598 |
+-----------+----------+----------+------------+
|         5 | a        |        7 |   0.014290 |
+-----------+----------+----------+------------+

ORACIÓN 2:
Gregorio! ¡Pobre hijo mío! ¿No comprendéis que necesito verle?», Gregorio
+-----------+----------+----------+------------+
|   Ranking | Token    |   Índice |   Atención |
+===========+==========+==========+========